# Week 12 Live Coding
## Spending the last \$50 million

Seven states, 93 electoral votes, and you need 44 of them. We are going to simulate the election 200,000 times, then hand out the money a million at a time.

Six things we will do:
1. Simulate the election and read your chance of reaching 270 off the picture
2. Turn the correlation between states off, and watch it drop
3. Ask which state actually decides it
4. Read the spending regression
5. Build the allocator
6. Price the candidate's Arizona request, then set the effect to zero

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

sw = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk12_presidential_allocation/data/swing_states.csv')
ad = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk12_presidential_allocation/data/ad_spending_effects.csv')

BASE = 226        # electoral votes you already hold
EV = sw['electoral_votes'].values
m0 = sw['margin'].values.astype(float)
sw

## Part 1: Simulate the election

You do not win a margin. You win or lose each state, and then you count.

Polls are wrong in two ways at once. There is a **national** miss that hits every state the same direction (2020 overstated Biden nearly everywhere), and a **state-specific** miss on top of it. From Week 8, the national part is the bigger one.

In [ ]:
rng = np.random.default_rng(2026)
N = 200_000
NAT_SD, ST_SD = 3.5, 2.0

national = rng.normal(0, NAT_SD, N)[:, None]        # one draw per election, all 7 states
state = rng.normal(0, ST_SD, (N, len(sw)))          # one draw per state per election

def simulate(margins):
    won = (margins[None, :] + national + state) > 0
    return BASE + (won * EV).sum(axis=1), won

evs, won = simulate(m0)
print('mean electoral votes:', round(evs.mean(), 1))
print('chance of reaching 270:', round(100 * (evs >= 270).mean(), 1), '%')

261.8 expected electoral votes and a 39.5% chance of winning. Those are different numbers and only one of them is the goal.

You never actually get 261.8. Look at what you do get.

In [ ]:
vals, counts = np.unique(evs, return_counts=True)
plt.figure(figsize=(9, 3.6))
plt.bar(vals, counts / N, width=4,
        color=['#0F4D92' if v >= 270 else '#555555' for v in vals])
plt.axvline(269.5, color='#B58900', lw=2)
plt.xlabel('Your electoral votes'); plt.ylabel('share of simulations')
plt.xlim(215, 325); plt.yticks([])
plt.show()

Two big spikes: losing all seven (226) and winning all seven (319). That is the national error at work. When the polls miss, they miss everywhere at once.

## Part 2: What if states moved independently?

Turn the shared national error off and give each state the same total uncertainty on its own.

In [ ]:
TOTAL_SD = np.sqrt(NAT_SD**2 + ST_SD**2)
print('one state total sd:', round(TOTAL_SD, 2), 'points')

# its own generator, so re-running this cell alone gives the same answer
indep = np.random.default_rng(33).normal(0, TOTAL_SD, (N, len(sw)))
evs_indep = BASE + (((m0[None, :] + indep) > 0) * EV).sum(axis=1)

print('errors move together :', round(100 * (evs >= 270).mean(), 1), '%')
print('errors independent   :', round(100 * (evs_indep >= 270).mean(), 1), '%')

Six points, and the sign is worth a beat. You are behind in five of seven states. Independent errors would need five separate lucky breaks. One shared error gives you a single shot at all of them.

Correlation helps whoever is behind. It also means a forecast that ignores it will tell a trailing candidate they have less of a chance than they do.

## Part 3: Which state decides it?

A state is **decisive** when it flips the outcome: you won it and needed it, or you lost it and it would have saved you.

In [ ]:
rows = []
for i, s in enumerate(sw['state']):
    without = evs - won[:, i] * EV[i]          # take the state away if you won it
    decisive = ((without < 270) & (without + EV[i] >= 270)).mean()
    rows.append((s, EV[i], m0[i], round(100 * decisive, 1)))

pd.DataFrame(rows, columns=['state', 'EVs', 'margin', 'decides it (%)']
             ).sort_values('decides it (%)', ascending=False)

Nevada is nearly tied and comes last. Six electoral votes rarely swing anything.

## Part 4: Where the analytics director's 0.07 came from

She took sixty past campaigns and compared each one's net ad-spending advantage in a state to how that state's margin moved. We need her number and her range to build anything, so here they are.

In [ ]:
reg = smf.ols('margin_shift_pp ~ spend_advantage_m', data=ad).fit()
print(reg.summary().tables[1])
COEF = reg.params['spend_advantage_m']
lo, hi = reg.conf_int().loc['spend_advantage_m']
print()
print('$50M in one state moves it', round(50 * COEF, 2), 'points')
print('   across the interval:', round(50 * lo, 2), 'to', round(50 * hi, 2))

**Nobody randomized any of this**, so do not read it as an experiment. Campaigns spend where it is already close, which means this number credits the ad buy for closeness it did not cause. It is almost certainly too big. We use it because it is the number in the room, and Part 6 prices what happens if it is wrong.

## Part 5: The allocator

Two ingredients. **Diminishing returns**, so the tenth million in a state does less than the first. And a rule for handing out the money.

In [ ]:
SCALE = 40.0
CEILING = COEF * SCALE          # a state cannot be moved more than this, ever

def gain(dollars):
    return CEILING * (1 - np.exp(-COEF * dollars / CEILING))

for d in [1, 10, 25, 50]:
    print('$' + str(d) + 'M buys', round(gain(d), 2), 'points  (a straight line would say',
          round(COEF * d, 2), ')')

In [ ]:
def margins_of(alloc):
    return m0 + np.array([gain(d) for d in alloc])

def chance_of_270(margins):
    ev = BASE + (((margins[None, :] + national + state) > 0) * EV).sum(axis=1)
    return (ev >= 270).mean()

def expected_evs(margins):
    from scipy.stats import norm
    return BASE + (norm.cdf(margins / TOTAL_SD) * EV).sum()


Now one round by hand, before we loop it. Ask what a single million would do in each of the seven states.

In [ ]:
# Round 1, by hand. What would the very first million do in each state?
alloc = np.zeros(len(sw))
before = chance_of_270(margins_of(alloc))

for i in range(len(sw)):
    trial = alloc.copy()
    trial[i] += 1
    gain_pts = 100 * (chance_of_270(margins_of(trial)) - before)
    print(f"{sw['state'][i]}  {EV[i]:2d} EVs  margin {m0[i]:+5.1f}"
          f"  ->  the next $1M buys {gain_pts:+.3f} points")


Pennsylvania wins by a wide margin: big **and** close. Nevada is closer than Georgia, Arizona and North Carolina and comes last, because six electoral votes rarely change the answer.

Pennsylvania goes on winning for twenty-two rounds. Then it stops, and the reason is not only that Pennsylvania is filling up.

In [ ]:
# Why does Pennsylvania ever stop winning? Watch all three at once.
for d in [0, 9, 19, 22]:
    a = np.zeros(len(sw)); a[0] = d
    base = chance_of_270(margins_of(a))
    row = ''
    for i in [0, 5, 3]:                      # PA, WI, MI
        trial = a.copy(); trial[i] += 1
        step = 100 * (chance_of_270(margins_of(trial)) - base)
        row += f"   {sw['state'][i]} {step:+.3f}"
    print(f"${d:2d}M already in PA:" + row)


Pennsylvania's number falls, which is diminishing returns. But **Wisconsin's and Michigan's rise.** Once Pennsylvania is probably yours you are holding 245 electoral votes, so you need 25 more, and Michigan 15 plus Wisconsin 10 is exactly 25. Buying Pennsylvania is what made them worth having.

At \$22M the lines cross. Now do all fifty rounds, twice: once maximizing your chance of reaching 270, once maximizing expected electoral votes.

In [ ]:
def allocate(objective, budget=50):
    alloc = np.zeros(len(sw))
    for _ in range(budget):
        scores = []
        for i in range(len(sw)):
            trial = alloc.copy(); trial[i] += 1
            scores.append(objective(margins_of(trial)))
        alloc[np.argmax(scores)] += 1
    return alloc

plan_270 = allocate(chance_of_270)
plan_ev = allocate(expected_evs)

pd.DataFrame({'state': sw['state'], 'EVs': EV,
              'maximize P(270)': plan_270.astype(int),
              'maximize expected EVs': plan_ev.astype(int)})


Two plans, same money, same model. The only thing that changed is what you told it to maximize.

In [ ]:
for name, plan in [('do nothing', np.zeros(len(sw))),
                   ('maximize expected EVs', plan_ev),
                   ('maximize P(270)', plan_270)]:
    m = margins_of(plan)
    print(name.ljust(24), 'expected EVs', round(expected_evs(m), 1),
          '| chance of 270', str(round(100 * chance_of_270(m), 1)) + '%')

The expected-EV plan wins on expected electoral votes and loses the election more often.

Look at what the P(270) plan bought: Pennsylvania 19, Michigan 15, Wisconsin 10. **That is 44, which is exactly the number you need.** Expected electoral votes cannot see a threshold, so it happily buys electoral votes in Georgia and North Carolina that only pay off in worlds where you were already winning.

## Part 6: What the candidate wants, and what if it is all zero

In [ ]:
def allocate_with_arizona(budget=50, arizona=20):
    alloc = np.zeros(len(sw))
    alloc[list(sw['state']).index('AZ')] = arizona
    for _ in range(budget - arizona):
        scores = []
        for i in range(len(sw)):
            trial = alloc.copy(); trial[i] += 1
            scores.append(chance_of_270(margins_of(trial)))
        alloc[np.argmax(scores)] += 1
    return alloc

az_plan = allocate_with_arizona()
print(', '.join(s + ' $' + str(int(d)) + 'M' for s, d in zip(sw['state'], az_plan) if d > 0))
best = chance_of_270(margins_of(plan_270))
withaz = chance_of_270(margins_of(az_plan))
print('best plan  :', round(100 * best, 1), '%')
print('with $20M in Arizona:', round(100 * withaz, 1), '%')
print('the request costs', round(100 * (best - withaz), 1), 'points of win probability')

That is a price, not a veto. It is not obviously too much to pay for a candidate who campaigns better somewhere she likes.

Now the question your polling director keeps asking. Kalla and Broockman's best guess for general-election persuasion is that it moves about 1 in 800 voters. Set the coefficient to zero and run the whole thing again.

In [ ]:
for c in [COEF, lo, 0.0]:
    if c <= 0:
        m = m0.copy()                       # spending buys nothing
    else:
        ceiling = c * SCALE
        m = m0 + ceiling * (1 - np.exp(-c * plan_270 / ceiling))
    print('if the true effect is', round(c, 3),
          '-> the same plan gives you', str(round(100 * chance_of_270(m), 1)) + '%')

\$50 million buys somewhere between eight points of win probability and nothing at all, and the best evidence in the field sits at the bottom of that range.

You still have to decide by Friday. That is the job.